---
title: "Exercise 4B: Xenium Normalization"
format: html
editor: source
editor_options:
  chunk_output_type: console
jupyter: ir
---

## Normalization and scaling for Xenium

In this exercise, we normalize the Xenium `SpatialFeatureExperiment` from Exercise 1B. The observations are segmented cells, and the measured features come from a targeted Xenium panel rather than a whole-transcriptome assay.

This means that normalization is useful for visualization and exploratory analysis, but the interpretation differs from Visium HD normalization.

## Learning objectives

By the end of this exercise, you will be able to:

- Add log-normalized values to a Xenium `SpatialFeatureExperiment`.
- Handle zero-count cells before normalization.
- Inspect size factors and assay names.
- Compare raw and log-normalized marker patterns.
- Scale selected targeted-panel features.

## Libraries

In [ ]:
#| warning: false
#| output: false

library(SpatialFeatureExperiment)
library(scuttle)
library(scater)
library(Voyager)
library(patchwork)
library(HDF5Array)
library(alabaster.sfe)

## Load the Xenium object

We start with the QC-filtered Xenium object saved at the end of Exercise 2B.

In [ ]:
# Reload the SpatialFeatureExperiment object if not in the R session already
sfe <- readObject("results/day1/01.2b_sfe_xenium/")
sfe

## Log-normalization

The Xenium object starts with raw counts. Some segmented cells may have zero counts for this targeted panel in the selected region. These cells cannot receive positive size factors, so we remove them before log-normalization.

We use the `scuttle::logNormCounts()` function to compute size factors and add a `logcounts` assay.

::: callout-note
This function will deprecate soon and replaced by the faster `scrapper::normalizeRnaCounts.se()` function.
:::

In [ ]:
assayNames(sfe)

cell_totals <- colSums(counts(sfe))
table(cell_totals == 0)

sfe <- sfe[, cell_totals > 0]

sfe <- logNormCounts(sfe)

assayNames(sfe)
summary(sizeFactors(sfe))

::: callout-important
## Exercise 1

How many zero-count cells were removed before normalization? Why can zero-count cells be a problem for size-factor normalization?
:::

::: {.callout-tip collapse="true"}
## Answer

In [ ]:
table(cell_totals == 0)

Cells with zero total counts would have zero size factors, but size factors must be positive for log-normalization.
:::

::: callout-important
## Exercise 2

How is normalization of Xenium data conceptually different from normalization of Visium HD data?
:::

::: {.callout-tip collapse="true"}
## Answer

Visium HD binned data are fixed spatial areas. Differences in total counts can reflect tissue density, local capture efficiency, and biology within each cell.

In Visium HD or Xenium segmented cells, differences in total counts can reflect cell size, segmentation, local detection efficiency

Specific to the Xenium is the targeted panel design. Because Xenium measures a selected panel of genes, normalized values are useful but should not be interpreted exactly like whole-transcriptome logcounts. Indeed the panels are often create to study specific cell-types in the tissue section, which breaks some assumpitons of the normalization methods. See this interesting paper about this issue: https://link.springer.com/article/10.1186/s13059-024-03303-w
:::

We can compare raw and log-normalized spatial patterns for a marker present in the Xenium panel.

In [ ]:
p_counts <- plotSpatialFeature(sfe, "PIGR", exprs_values = "counts") +
  ggtitle("PIGR counts")

p_logcounts <- plotSpatialFeature(sfe, "PIGR", exprs_values = "logcounts") +
  ggtitle("PIGR logcounts")

p_counts + p_logcounts

::: callout-important
## Exercise 3

How does the `PIGR` pattern change after log-normalization? Does normalization remove all spatial structure?
:::

## Scaling selected features

Scaling centers and rescales features so that they can be compared on a common scale. This is useful for visualization and multivariate methods, but here the features are from a targeted panel.

In [ ]:
marker_genes <- c("PIGR", "CEACAM5", "MUC17", "OLFM4")
marker_genes <- intersect(marker_genes, rownames(sfe))
marker_genes

In [ ]:
scale_rows <- function(x) {
  t(base::scale(t(x)))
}

xenium_scaled <- scale_rows(as.matrix(logcounts(sfe)[marker_genes, ]))
xenium_scaled[, 1:5]

::: callout-important
## Exercise 4

What does a positive scaled value mean for a gene in one cell? Why should scaling be interpreted in the context of the targeted Xenium panel?
:::

::: {.callout-tip collapse="true"}
## Answer

A positive scaled value means that the cell has expression above the average for that gene, relative to the variation of that gene across the selected cells. Because Xenium measures a targeted panel, scaling compares genes and cells within that panel rather than across a full transcriptome.
:::

## Save normalized object

We save the normalized Xenium object for later optional comparisons.

In [ ]:
dir.create("results/day2", showWarnings = FALSE, recursive = TRUE)
alabaster.sfe::saveObject(sfe, "results/day2/02.1b_sfe_xenium")

Clear your environment:

In [ ]:
#| eval: false
rm(list = ls())
gc()
.rs.restartR()

::: callout-important
**Key Takeaways:**

- Xenium normalization should be performed separately from Visium HD normalization.
- Zero-count cells must be handled before size-factor normalization.
- Xenium normalized values are useful, but should be interpreted in the context of segmented cells and a targeted panel.
:::